# More stacks: 4 and 5 layers, and a heterostructure

Companion to `yaml_multilayer_demo.ipynb`. Three stack specs from `examples/`:

| spec | what it exercises |
|---|---|
| `bto_4layer_demo.yaml` | two bilayer blocks combined as rigid units |
| `bto_5layer_demo.yaml` | carry-forward over two rounds (L5 joins in round 3) |
| `pto_sto_pto_demo.yaml` | a heterostructure, and why `max_layer_residual` matters |

In [1]:
import os
os.environ["TQDM_DISABLE"] = "1"
from pathlib import Path
import numpy as np
from MLM.spec import run_stack_spec
from MLM.index import load_index, find, lineage
from MLM.build import write_structure, build_atoms
from MLM.index import get
REPO = Path("..")
COLUMNS = ["id", "n_layers", "twist", "residual", "cumulative_residual", "max_layer_residual", "cell_area", "passed_filter"]

## 1. Four layers

Plan: round 1 = (L1 + L2), (L3 + L4); round 2 = (L1-2) + (L3-4). Both round-1 pairs give the
identical bilayer recipe, so it is stored once and appears twice as a parent.

In [2]:
four = run_stack_spec(REPO / "examples/bto_4layer_demo.yaml", base_dir=REPO, log=None)
print(four.summary())

run directory: ../runs/bto_4layer_demo
round 1: L1 + L2 -> L1-2, L3 + L4 -> L3-4
round 2: L1-2 + L3-4 -> L1-4
round 1: L1 + L2 -> L1-2: 1 x 1 = 1 searches -> 1 candidates, 1 passed filters (0.2 s)
round 1: L3 + L4 -> L3-4: 1 x 1 = 1 searches -> 1 candidates, 1 passed filters (0.2 s)
round 2: L1-2 + L3-4 -> L1-4: 1 x 1 = 1 searches -> 4 candidates, 2 passed filters (1.6 s)
full-stack candidates passing filters: 2


In [3]:
best4 = find(four.run_dir, n_layers=4, passed_filter=True)
best4[COLUMNS]

,id,n_layers,twist,residual,cumulative_residual,max_layer_residual,cell_area,passed_filter
5,bto_4_22.62_22.62_36.87_c30654,4,36.87,0.000057,0.000057,0.000133,1035.003146,True
2,bto_4_22.62_22.62_30.51_eb69f1,4,30.51,0.000481,0.000481,0.000481,13455.040898,True


In [4]:
four_id = best4["id"].iloc[0]
print(lineage(four.run_dir, four_id))
print("layer rotations:", np.round(get(four.run_dir, four_id).layer_rotations, 3))

bto_4_22.62_22.62_36.87_c30654  (4 layers)  twist=36.8700  residual=5.75e-05
    bto_2_22.62_f0fd88  (2 layers)  twist=22.6200  residual=3.39e-05
        bto_1_65687f  (1 layer)
        bto_1_65687f  (1 layer)
    bto_2_22.62_f0fd88  (2 layers)  twist=22.6200  residual=3.39e-05
        bto_1_65687f  (1 layer)
        bto_1_65687f  (1 layer)
layer rotations: [ 0.   22.62 36.87 59.49]


Layers 3 and 4 keep their relative twist of 22.62° — the upper bilayer was rotated as a block.

## 2. Five layers (carry-forward)

L5 has no partner in rounds 1 and 2 and is carried forward unchanged.
The dry run shows the plan before anything is searched:

In [5]:
_ = run_stack_spec(REPO / "examples/bto_5layer_demo.yaml", base_dir=REPO, dry_run=True)

DRY RUN — nothing searched
round 1: L1 + L2 -> L1-2, L3 + L4 -> L3-4, carry L5
round 2: L1-2 + L3-4 -> L1-4, carry L5
round 3: L1-4 + L5 -> L1-5
round 1: L1 + L2 -> L1-2: 1 x 1 = 1 searches
round 1: L3 + L4 -> L3-4: 1 x 1 = 1 searches
round 1: carry L5 (1 candidates)
round 2: L1-2 + L3-4 -> L1-4: ? x ? = ? searches (known once the previous round has run)
round 2: carry L5 (1 candidates)
round 3: L1-4 + L5 -> L1-5: ? x 1 = ? searches (known once the previous round has run)


In [6]:
five = run_stack_spec(REPO / "examples/bto_5layer_demo.yaml", base_dir=REPO, log=None)
print(five.summary())

run directory: ../runs/bto_5layer_demo
round 1: L1 + L2 -> L1-2, L3 + L4 -> L3-4, carry L5
round 2: L1-2 + L3-4 -> L1-4, carry L5
round 3: L1-4 + L5 -> L1-5
round 1: L1 + L2 -> L1-2: 1 x 1 = 1 searches -> 1 candidates, 1 passed filters (0.2 s)
round 1: L3 + L4 -> L3-4: 1 x 1 = 1 searches -> 1 candidates, 1 passed filters (0.2 s)
round 1: carry L5 (1 candidates)
round 2: L1-2 + L3-4 -> L1-4: 1 x 1 = 1 searches -> 1 candidates, 1 passed filters (0.2 s)
round 2: carry L5 (1 candidates)
round 3: L1-4 + L5 -> L1-5: 1 x 1 = 1 searches -> 5 candidates, 2 passed filters (13.0 s)
full-stack candidates passing filters: 2


In [7]:
table5 = load_index(five.run_dir)
table5[COLUMNS]

,id,n_layers,twist,residual,cumulative_residual,max_layer_residual,cell_area,passed_filter
0,bto_1_65687f,1,NaN,NaN,NaN,NaN,1.592313e+01,True
1,bto_2_22.62_f0fd88,2,22.62,0.000034,0.000034,0.000034,2.070006e+02,True
2,bto_4_22.62_22.62_36.87_5f8754,4,36.87,0.000057,0.000057,0.000133,1.035003e+03,True
3,bto_5_22.62_22.62_36.87_20.61_ec6ee0,5,20.61,0.000862,0.000862,0.000862,2.587508e+04,True
4,bto_5_22.62_22.62_36.87_22.62_bae36b,5,22.62,0.000107,0.000107,0.000189,2.070006e+03,True
5,bto_5_22.62_22.62_36.87_24.19_c64544,5,24.19,0.001747,0.001747,0.001747,4.243513e+04,False
6,bto_5_22.62_22.62_36.87_24.33_3a5fbc,5,24.33,0.001589,0.001589,0.006316,2.323582e+06,False
7,bto_5_22.62_22.62_36.87_24.63_d9409b,5,24.63,0.000837,0.000837,0.005374,1.681880e+06,False


`passed_filter = False` rows are kept on disk: the spec also asks for
`max_layer_residual ≤ 1e-3`. Look at the rejected 5-layer cells: their step `residual` is
below 1e-3 Å, but measured against every layer the error is several times larger.

In [8]:
five_id = find(five.run_dir, n_layers=5, passed_filter=True)["id"].iloc[0]
print(lineage(five.run_dir, five_id))
path = write_structure(five.run_dir, five_id)
built = build_atoms(get(five.run_dir, five_id), search_dirs=[REPO])
print(path, "| atoms per layer:", built.layer_counts, "expected:", built.expected_counts)

bto_5_22.62_22.62_36.87_22.62_bae36b  (5 layers)  twist=22.6200  residual=1.07e-04
    bto_4_22.62_22.62_36.87_5f8754  (4 layers)  twist=36.8700  residual=5.75e-05
        bto_2_22.62_f0fd88  (2 layers)  twist=22.6200  residual=3.39e-05
            bto_1_65687f  (1 layer)
            bto_1_65687f  (1 layer)
        bto_2_22.62_f0fd88  (2 layers)  twist=22.6200  residual=3.39e-05
            bto_1_65687f  (1 layer)
            bto_1_65687f  (1 layer)
    bto_1_65687f  (1 layer)


../runs/bto_5layer_demo/structures/bto_5_22.62_22.62_36.87_22.62_bae36b.vasp | atoms per layer: [910, 910, 910, 910, 910] expected: [910, 910, 910, 910, 910]


## 3. Heterostructure PTO / STO / PTO

PTO (a = 3.880 Å) and STO (a = 3.913 Å) are never exactly commensurate; the bilayer cells of
`Bi-layer_structure_finder.ipynb` (dtol 9e-3) have residuals of 0.008–0.03 Å.

In [9]:
hetero = run_stack_spec(REPO / "examples/pto_sto_pto_demo.yaml", base_dir=REPO, log=None)
print(hetero.summary())

run directory: ../runs/pto_sto_pto_demo
round 1: L1 + L2 -> L1-2, carry L3
round 2: L1-2 + L3 -> L1-3
round 1: L1 + L2 -> L1-2: 1 x 1 = 1 searches -> 2 candidates, 2 passed filters (0.4 s)
round 1: carry L3 (1 candidates)
round 2: L1-2 + L3 -> L1-3: 2 x 1 = 2 searches -> 21 candidates, 5 passed filters (13.8 s)
full-stack candidates passing filters: 5


In [10]:
bilayers = find(hetero.run_dir, n_layers=2)
bilayers[COLUMNS]

,id,n_layers,twist,residual,cumulative_residual,max_layer_residual,cell_area,passed_filter
2,pto_sto_2_28.99_8b52f9,2,28.99,0.008259,0.008259,0.008259,2724.217053,True
3,pto_sto_2_29.74_57fcbb,2,29.74,0.023813,0.023813,0.023813,978.309991,True


The trilayer step residuals look harmless, but the cell's error with respect to the STO
layer (`max_layer_residual`) is several times larger (median ≈ 5× here), because the trilayer cell is a large
integer multiple of an already-imperfect bilayer cell. `cumulative_residual` (the agreed
"max over steps" rule) does not see this:

In [11]:
tri = find(hetero.run_dir, n_layers=3)
print(f"{len(tri)} trilayer candidates, {tri['passed_filter'].sum()} pass max_layer_residual <= 0.05 Å")
ratio = tri["max_layer_residual"] / tri["cumulative_residual"]
print(f"max_layer_residual / cumulative_residual: median {ratio.median():.1f}, max {ratio.max():.0f}")
tri[tri["passed_filter"]].sort_values("cell_area")[COLUMNS].head(10)

21 trilayer candidates, 5 pass max_layer_residual <= 0.05 Å
max_layer_residual / cumulative_residual: median 5.0, max 8


,id,n_layers,twist,residual,cumulative_residual,max_layer_residual,cell_area,passed_filter
15,pto_sto_pto_3_29.74_22.62_308ad9,3,22.62,0.000104,0.023813,0.033676,1956.619983,True
22,pto_sto_pto_3_29.74_36.87_c8f8f3,3,36.87,0.000079,0.023813,0.033676,1956.619983,True
21,pto_sto_pto_3_28.99_36.87_ba0f76,3,36.87,0.000208,0.008259,0.018469,13621.085265,True
14,pto_sto_pto_3_28.99_22.62_90d4c3,3,22.62,0.000444,0.008259,0.029780,35414.821689,True
11,pto_sto_pto_3_28.99_16.26_19960d,3,16.26,0.000932,0.008259,0.041297,68105.426325,True


In [12]:
hetero_id = tri[tri["passed_filter"]].sort_values("cell_area")["id"].iloc[0]
print(lineage(hetero.run_dir, hetero_id))
record = get(hetero.run_dir, hetero_id)
print("materials:", record.material, " rotations:", np.round(record.layer_rotations, 2))
built = build_atoms(record, search_dirs=[REPO])
print("atoms per layer:", built.layer_counts, "expected:", built.expected_counts, "| composition:", built.composition)
print(write_structure(hetero.run_dir, hetero_id))

pto_sto_pto_3_29.74_22.62_308ad9  (3 layers)  twist=22.6200  residual=1.04e-04
    pto_sto_2_29.74_57fcbb  (2 layers)  twist=29.7400  residual=2.38e-02
        pto_1_e344d9  (1 layer)
        sto_1_6bab52  (1 layer)
    pto_1_e344d9  (1 layer)
materials: ['pto', 'sto', 'pto']  rotations: [ 0.   29.74 22.62]
atoms per layer: [910, 896, 910] expected: [910, 896, 910] | composition: {'O': 1552, 'Pb': 520, 'Sr': 256, 'Ti': 388}


../runs/pto_sto_pto_demo/structures/pto_sto_pto_3_29.74_22.62_308ad9.vasp
